In [2]:
import os
from pathlib import Path
import pandas as pd

# Pastikan direktori kerja berada di akar proyek (mlops-kredit-sumatera)
if Path.cwd().name == "notebooks":
    os.chdir("..")

df = pd.read_csv("data/raw/kredit_sumatera.csv")

# Pertanyaan 1: seberapa tidak seimbang targetnya?
print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")

# Pertanyaan 2: apakah populasi berubah sepanjang waktu?
print("\nProporsi segmen per tahun:")
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize="index").round(3))

# Pertanyaan 3: kolom mana yang punya nilai hilang, dan mengapa?
hilang = df.isna().mean().sort_values(ascending=False)
print("\nKolom dengan nilai hilang:")
print(hilang[hilang > 0].round(4))
print("\nNilai hilang rata_saldo_90h per segmen:")
print(df.groupby("segmen")["rata_saldo_90h"].apply(lambda s: s.isna().mean()).round(3))

# Pertanyaan 4: adakah kolom yang mencurigakan?
print("\nKeterisian status_restrukturisasi per target:")
print(df.groupby("gagal_bayar")["status_restrukturisasi"].apply(lambda s: s.notna().mean()))

Tingkat gagal bayar: 8.30%

Proporsi segmen per tahun:
segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231

Kolom dengan nilai hilang:
status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64

Nilai hilang rata_saldo_90h per segmen:
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64

Keterisian status_restrukturisasi per target:
gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


In [5]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Daftar struktur kolom secara vertikal
print(f"Total kolom: {len(df.columns)} kolom")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<26} ({df[col].dtype})")

# 2. Fitur sesuai acuan modul (membuang kolom ID, tanggal/periode, dan kolom bocor)
df["rasio_angsuran"] = df.angsuran_bulanan / df.penghasilan_bulanan
num = ["usia", "tanggungan", "lama_usaha_tahun", "penghasilan_bulanan",
       "ragam_penghasilan", "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
       "rasio_angsuran", "rata_saldo_90h", "jumlah_tunggakan_12b"]

X = df[num].join(pd.get_dummies(df[["segmen", "wilayah", "kanal"]]))
y = df.gagal_bayar

# 3. Pembagian MENURUT WAKTU (Out-of-Time): latih <= 2023-12, uji > 2024-06
latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                   max_iter=250, random_state=42)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
auc_oot = roc_auc_score(y[uji], skor)

# 4. Pembagian ACAK (80:20) sebagai pembanding
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                    max_iter=250, random_state=42).fit(Xa, ya)
auc_acak = roc_auc_score(yb, m2.predict_proba(Xb)[:, 1])

print("\n" + "=" * 46)
print("PERBANDINGAN EVALUASI MODEL GARIS DASAR")
print("=" * 46)
print(f"AUC out-of-time (OOT) : {auc_oot:.4f}")
print(f"AUC pembagian acak    : {auc_acak:.4f}")
print(f"Selisih kejujuran     : {auc_oot - auc_acak:+.4f}")
print("=" * 46)

Total kolom: 19 kolom
   1. pengajuan_id               (object)
   2. tanggal_pengajuan          (object)
   3. periode                    (object)
   4. segmen                     (object)
   5. wilayah                    (object)
   6. kanal                      (object)
   7. usia                       (int64)
   8. tanggungan                 (int64)
   9. lama_usaha_tahun           (float64)
  10. penghasilan_bulanan        (float64)
  11. ragam_penghasilan          (float64)
  12. plafon_diajukan            (float64)
  13. tenor_bulan                (int64)
  14. bunga_tahunan              (float64)
  15. angsuran_bulanan           (float64)
  16. rata_saldo_90h             (float64)
  17. jumlah_tunggakan_12b       (int64)
  18. gagal_bayar                (int64)
  19. status_restrukturisasi     (object)

PERBANDINGAN EVALUASI MODEL GARIS DASAR
AUC out-of-time (OOT) : 0.7744
AUC pembagian acak    : 0.7631
Selisih kejujuran     : +0.0112
